# Orquestador Silver, DLQ y Gold

Ejecuta la creación de schemas, las cuatro transformaciones Silver/DLQ y Gold en orden. Bronze se lee solamente; la ingesta no pertenece a este flujo.

In [0]:
import json
import time
import uuid

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("audit_schema", "audit")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("run_ingestion", "false")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("timeout_seconds", "600")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
audit_schema = dbutils.widgets.get("audit_schema")
gold_schema = dbutils.widgets.get("gold_schema")
run_ingestion = dbutils.widgets.get("run_ingestion").lower()
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())

if run_ingestion != "false":
    raise ValueError("run_ingestion debe ser false: Bronze es de solo lectura")
cutoff_us = dbutils.widgets.get("cutoff_us") or str(int(time.time() * 1_000_000))
if not cutoff_us.isdigit():
    raise ValueError("cutoff_us debe ser un epoch en microsegundos")

base_args = {
    "catalog": catalog,
    "bronze_schema": bronze_schema,
    "silver_schema": silver_schema,
    "dlq_schema": dlq_schema,
    "audit_schema": audit_schema,
    "gold_schema": gold_schema,
    "cutoff_us": cutoff_us,
    "run_id": run_id,
}
timeout_seconds = int(dbutils.widgets.get("timeout_seconds"))
NOTEBOOKS = [
    "../src/transformation/silver_crear_schemas",
    "../src/transformation/silver_games",
    "../src/transformation/silver_game_reviews",
    "../src/transformation/silver_player_counts_monthly",
    "../src/transformation/silver_twitch_activity_monthly",
    "../src/gold",
]
results = {}
for notebook in NOTEBOOKS:
    print(f"Ejecutando: {notebook}")
    raw_result = dbutils.notebook.run(notebook, timeout_seconds, base_args)
    result = json.loads(raw_result)
    if result.get("status") != "completed":
        raise RuntimeError(f"La transformación {notebook} no terminó correctamente: {result}")
    results[notebook] = result
    print(f"Completado: {notebook}")

dbutils.notebook.exit(json.dumps({"status": "completed", "run_id": run_id, "cutoff_us": int(cutoff_us), "tasks": results}))